In [7]:

import websocket
import json
import csv
import datetime
import os
import pandas as pd
import threading
import time
from sqlalchemy import create_engine
import urllib.parse

In [8]:

WEBSOCKET_URL = "wss://ws.bitget.com/v2/ws/public"
INSTRUMENT_IDS = ["SOLUSDT", "BTCUSDT", "ETHUSDT"]  
CSV_FILE_NAME = "data.csv"


In [ ]:

WEBSOCKET_URL = "wss://ws.bitget.com/v2/ws/public"
INSTRUMENT_IDS = ["SOLUSDT", "BTCUSDT", "ETHUSDT"]  
CSV_FILE_NAME = "data.csv"
CANDLESTICK_CSV_FILE = "candlestick_data.csv"
TIMEFRAMES = ["candle1m", "candle5m", "candle15m", "candle1H"]

In [ ]:

def tao_file_csv():
    header = [
        "thoi_gian", "gia", "gia_mua", "gia_ban", "khoi_luong_24h", "high24h", "low24h", "instId",
        "best_purchase_price", "best_sale_price", "change24h", "funding_rate", "next_funding_time",
        "mark_price", "index_price", "holding_amount", "base_volume", "quote_volume", "open_utc",
        "symbol_type", "symbol", "delivery_price", "open24h", "timestamp_api", "action"
    ]
    
    if not os.path.exists(CSV_FILE_NAME) or os.path.getsize(CSV_FILE_NAME) == 0:
        with open(CSV_FILE_NAME, mode='w', newline='', encoding='utf-8') as file:
            writer = csv.writer(file)
            writer.writerow(header)
        print(f"Đã tạo file CSV với {len(header)} cột: {CSV_FILE_NAME}")
    else:
        print(f"File CSV đã tồn tại: {CSV_FILE_NAME}")

tao_file_csv()

Đã tạo file CSV với 25 cột: data.csv


In [ ]:

def tao_file_candlestick_csv():
    candlestick_header = [
        "thoi_gian", "start_time", "open_price", "high_price", "low_price", "close_price",
        "base_volume", "quote_volume", "usdt_volume", "instId", "timeframe", "action", "timestamp_api"
    ]
    
    if not os.path.exists(CANDLESTICK_CSV_FILE) or os.path.getsize(CANDLESTICK_CSV_FILE) == 0:
        with open(CANDLESTICK_CSV_FILE, mode='w', newline='', encoding='utf-8') as file:
            writer = csv.writer(file)
            writer.writerow(candlestick_header)
        print(f"Đã tạo file Candlestick CSV với {len(candlestick_header)} cột: {CANDLESTICK_CSV_FILE}")
    else:
        print(f"File Candlestick CSV đã tồn tại: {CANDLESTICK_CSV_FILE}")

# Gọi function tạo file
tao_file_candlestick_csv()

Đã tạo file Candlestick CSV với 13 cột: candlestick_data.csv


In [ ]:

all_ticker_data = []
all_candlestick_data = []

def on_open_enhanced(ws):
    print("Đã kết nối thành công")
    
    # Subscribe to ticker data cho SPOT
    for inst_id in INSTRUMENT_IDS:
        ticker_message = {
            "op": "subscribe",
            "args": [
                {    
                    "instType": "SPOT",  # Thay đổi từ USDT-FUTURES thành SPOT
                    "channel": "ticker",
                    "instId": inst_id
                }
            ]
        }
        ws.send(json.dumps(ticker_message))
        print(f"Đang theo dõi ticker SPOT {inst_id}")
    
    # Subscribe to candlestick data cho SPOT  
    for inst_id in INSTRUMENT_IDS:
        for timeframe in TIMEFRAMES:
            candlestick_message = {
                "op": "subscribe",
                "args": [
                    {    
                        "instType": "SPOT",  # Thay đổi từ USDT-FUTURES thành SPOT
                        "channel": timeframe,
                        "instId": inst_id
                    }
                ]
            }
            ws.send(json.dumps(candlestick_message))
            print(f"Đang theo dõi candlestick SPOT {inst_id} - {timeframe}")
    
    print(f"Hoàn tất subscribe cho {len(INSTRUMENT_IDS)} coins SPOT với {len(TIMEFRAMES)} timeframes")

def on_message_enhanced(ws, message_str):
    global all_ticker_data, all_candlestick_data
    
    data = json.loads(message_str)
    
    
    if "arg" in data and "channel" in data["arg"]:
        channel = data["arg"]["channel"]
        
        if channel == "ticker":
            
            all_ticker_data.append(data)
            process_ticker_data(data)
            
        elif channel.startswith("candle"):
            
            all_candlestick_data.append(data)
            process_candlestick_data(data)

def process_ticker_data(data):
    """Xử lý dữ liệu ticker"""
    if "data" in data and data["data"]:
        ticker = data["data"][0]
        
        
        thoi_gian = datetime.datetime.now().isoformat()
        gia = ticker.get('lastPr')
        gia_mua = ticker.get('bidPr')
        gia_ban = ticker.get('askPr')
        khoi_luong_24h = ticker.get('volumeUsd24h')
        high24h = ticker.get('high24h')
        low24h = ticker.get('low24h')
        instId = ticker.get('instId')
        best_purchase_price = ticker.get('bidSz')
        best_sale_price = ticker.get('askSz')
        change24h = ticker.get('change24h')
        funding_rate = ticker.get('fundingRate')
        next_funding_time = ticker.get('nextFundingTime')
        mark_price = ticker.get('markPrice')
        index_price = ticker.get('indexPrice')
        holding_amount = ticker.get('holdingAmount')
        base_volume = ticker.get('baseVolume')
        quote_volume = ticker.get('quoteVolume')
        open_utc = ticker.get('openUtc')
        symbol_type = ticker.get('symbolType')
        symbol = ticker.get('symbol')
        delivery_price = ticker.get('deliveryPrice')
        open24h = ticker.get('open24h')
        timestamp_api = ticker.get('ts')
        action = data.get('action', 'unknown')
        
        print(f"TICKER {instId} | Giá: {gia} | Thay đổi 24h: {change24h}")
        
        
        with open(CSV_FILE_NAME, mode='a', newline='', encoding='utf-8') as file:
            writer = csv.writer(file)
            writer.writerow([
                thoi_gian, gia, gia_mua, gia_ban, khoi_luong_24h, high24h, low24h, instId,
                best_purchase_price, best_sale_price, change24h, funding_rate, next_funding_time,
                mark_price, index_price, holding_amount, base_volume, quote_volume, open_utc,
                symbol_type, symbol, delivery_price, open24h, timestamp_api, action
            ])

def process_candlestick_data(data):
    """Xử lý dữ liệu candlestick"""
    if "data" in data and data["data"]:
        candle = data["data"][0]  
        arg = data.get("arg", {})
        
        
        thoi_gian = datetime.datetime.now().isoformat()
        start_time = candle[0]  
        open_price = candle[1]   
        high_price = candle[2]   
        low_price = candle[3]  
        close_price = candle[4]  
        base_volume = candle[5]  
        quote_volume = candle[6] 
        usdt_volume = candle[7]  
        
        instId = arg.get('instId')
        timeframe = arg.get('channel')
        action = data.get('action', 'unknown')
        timestamp_api = data.get('ts')
        
        
        try:
            start_time_readable = datetime.datetime.fromtimestamp(int(start_time)/1000).isoformat()
        except:
            start_time_readable = start_time
        
        print(f"CANDLE {instId} {timeframe} | O:{open_price} H:{high_price} L:{low_price} C:{close_price}")
        
       
        with open(CANDLESTICK_CSV_FILE, mode='a', newline='', encoding='utf-8') as file:
            writer = csv.writer(file)
            writer.writerow([
                thoi_gian, start_time_readable, open_price, high_price, low_price, close_price,
                base_volume, quote_volume, usdt_volume, instId, timeframe, action, timestamp_api
            ])

def on_error(ws, error):
    print(f"Lỗi: {error}")

def on_close(ws, close_status_code, close_msg):
    print(f"Kết nối đã đóng")

In [ ]:

a = 60  # 1 phút
b = a * 60  # 1h
c = b * 24  # 1 ngày

def run_ws_enhanced():
    ws.run_forever(ping_interval=30, ping_timeout=10)

ws = websocket.WebSocketApp(WEBSOCKET_URL,
                          on_open=on_open_enhanced,
                          on_message=on_message_enhanced,
                          on_error=on_error,
                          on_close=on_close)

print("Bắt đầu kết nối đến Bitget (Ticker + Candlestick)...")
print(f"Ticker data sẽ được lưu vào: {CSV_FILE_NAME}")
print(f"Candlestick data sẽ được lưu vào: {CANDLESTICK_CSV_FILE}")
print("Nhấn Ctrl+C để dừng")

ws_thread = threading.Thread(target=run_ws_enhanced)
ws_thread.daemon = True
ws_thread.start()

run_duration = a

try:
    time.sleep(run_duration)
except KeyboardInterrupt:
    print("\nĐã dừng bằng Ctrl+C")

ws.close()
print(f"Đã ngắt kết nối sau {run_duration} giây.")
print(f"Ticker data: {len(all_ticker_data)} messages")
print(f"Candlestick data: {len(all_candlestick_data)} messages")

Bắt đầu kết nối đến Bitget (Ticker + Candlestick)...
Ticker data sẽ được lưu vào: data.csv
Candlestick data sẽ được lưu vào: candlestick_data.csv
Nhấn Ctrl+C để dừng
Đã kết nối thành công
Đang theo dõi ticker SOLUSDT
Đang theo dõi ticker BTCUSDT
Đang theo dõi ticker ETHUSDT
Đang theo dõi candlestick SOLUSDT - candle1m
Đang theo dõi candlestick SOLUSDT - candle5m
Đang theo dõi candlestick SOLUSDT - candle15m
Đang theo dõi candlestick SOLUSDT - candle1H
Đang theo dõi candlestick BTCUSDT - candle1m
Đang theo dõi candlestick BTCUSDT - candle5m
Đang theo dõi candlestick BTCUSDT - candle15m
Đang theo dõi candlestick BTCUSDT - candle1H
Đang theo dõi candlestick ETHUSDT - candle1m
Đang theo dõi candlestick ETHUSDT - candle5m
Đang theo dõi candlestick ETHUSDT - candle15m
Đang theo dõi candlestick ETHUSDT - candle1H
Hoàn tất subscribe cho 3 coins với 4 timeframes
CANDLE SOLUSDT candle1m | O:155.663 H:155.78 L:155.61 C:155.777
CANDLE SOLUSDT candle5m | O:153.008 H:153.18 L:152.501 C:152.569
CANDL